<a href="https://colab.research.google.com/github/ekkaiah/claude_repository/blob/main/entrenar_cotxes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrenament model reconeixement de cotxes

**Dataset:** Stanford Cars (16.185 imatges, 196 classes marca+model+any)  
**Model:** EfficientNet-B3  
**GPU:** T4 gratuita de Colab  
**Temps:** ~1-2 hores

> **IMPORTANT:** Ves a `Runtime → Change runtime type → T4 GPU` abans d'executar

In [1]:
# Instal·lar dependències
!pip install -q datasets timm torchvision Pillow tqdm

In [2]:
# Verificar GPU
import torch
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Model GPU: {torch.cuda.get_device_name(0)}')
else:
    print('ATENCIO: Sense GPU. Ves a Runtime > Change runtime type > T4 GPU')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

GPU: True
Model GPU: Tesla T4


In [3]:
# Descarregar Stanford Cars Dataset
from datasets import load_dataset
print('Descarregant Stanford Cars (~1.8GB)...')
dataset = load_dataset('tanganke/stanford_cars')
class_names = dataset['train'].features['label'].names
num_classes = len(class_names)
print(f'Train: {len(dataset["train"])} | Test: {len(dataset["test"])} | Classes: {num_classes}')
print('Exemples:', class_names[:5])

Descarregant Stanford Cars (~1.8GB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/504M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/485M [00:00<?, ?B/s]

...

Generating jpeg_compression split:   0%|          | 0/8041 [00:00<?, ? examples/s]

Generating motion_blur split:   0%|          | 0/8041 [00:00<?, ? examples/s]

Generating pixelate split:   0%|          | 0/8041 [00:00<?, ? examples/s]

Generating spatter split:   0%|          | 0/8041 [00:00<?, ? examples/s]

Train: 8144 | Test: 8041 | Classes: 196
Exemples: ['AM General Hummer SUV 2000', 'Acura RL Sedan 2012', 'Acura TL Sedan 2012', 'Acura TL Type-S 2008', 'Acura TSX Sedan 2012']


In [4]:
# DataLoaders
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

IMG_SIZE = 300
BATCH_SIZE = 32

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class CarsDS(Dataset):
    def __init__(self, ds, tf):
        self.ds, self.tf = ds, tf
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        x = self.ds[i]
        return self.tf(x['image'].convert('RGB')), x['label']

train_loader = DataLoader(CarsDS(dataset['train'], train_tf), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(CarsDS(dataset['test'],  val_tf),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Batches train: {len(train_loader)} | val: {len(val_loader)}')

Batches train: 255 | val: 252


In [5]:
# Carregar model EfficientNet-B3
import timm
import torch.nn as nn
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=num_classes)
model = model.to(device)
print(f'Parametres: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Parametres: 10,997,484


In [6]:
# Entrenament
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

EPOCHS = 80
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    loss_sum = 0
    for imgs, lbls in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), lbls)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    model.eval()
    ok = total = 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            _, pred = model(imgs).max(1)
            total += lbls.size(0)
            ok += pred.eq(lbls).sum().item()
    acc = 100. * ok / total
    scheduler.step()
    print(f'Epoch {epoch+1:02}/{EPOCHS} | Loss: {loss_sum/len(train_loader):.4f} | Acc: {acc:.2f}%')
    if acc > best_acc:
        best_acc = acc
        torch.save({'state_dict': model.state_dict(), 'class_names': class_names, 'num_classes': num_classes}, '/content/drive/MyDrive/model_cotxes.pth')
        print(f'  NOU MILLOR MODEL: {acc:.2f}%')
print(f'\nMillor precisio: {best_acc:.2f}%')

Epoch 1/20: 100%|██████████| 255/255 [02:45<00:00,  1.54it/s]


Epoch 01/20 | Loss: 3.4392 | Acc: 64.81%
  NOU MILLOR MODEL: 64.81%


Epoch 2/20: 100%|██████████| 255/255 [02:51<00:00,  1.49it/s]


Epoch 02/20 | Loss: 1.6442 | Acc: 78.68%
  NOU MILLOR MODEL: 78.68%


Epoch 3/20: 100%|██████████| 255/255 [02:51<00:00,  1.49it/s]


...

Epoch 18/20 | Loss: 0.8684 | Acc: 93.00%


Epoch 19/20: 100%|██████████| 255/255 [02:51<00:00,  1.49it/s]


Epoch 19/20 | Loss: 0.8669 | Acc: 93.00%


Epoch 20/20: 100%|██████████| 255/255 [02:51<00:00,  1.49it/s]


Epoch 20/20 | Loss: 0.8662 | Acc: 92.85%

Millor precisio: 93.07%


In [13]:
# Prova amb una imatge
from google.colab import files
from PIL import Image
import torch

uploaded = files.upload()  # selecciona prova_mercedes.jpg del teu Mac

img = Image.open(list(uploaded.keys())[0]).convert('RGB')
t = val_tf(img).unsqueeze(0).to(device)
model.eval()
with torch.no_grad():
    probs = torch.softmax(model(t), dim=1)
    vals, idxs = probs.topk(3)
for p, i in zip(vals[0], idxs[0]):
    print(f'  {class_names[i]:55s} {p.item()*100:.1f}%')


Saving Macan 1.jpg to Macan 1.jpg
  Acura Integra Type R 2001                               5.2%
  BMW 3 Series Sedan 2012                                 3.1%
  Cadillac SRX SUV 2012                                   2.9%


In [12]:
# Descarregar model
from google.colab import files
files.download('model_cotxes.pth')
print('Model descarregat!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model descarregat!
